# Chatbot with streaming

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

## Instantiate the Sarvam AI Client

In [3]:
from sarvamai import SarvamAI

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Use Case Definition

Make the chatbot interact with customers, online helping them:

1. Check the Account Balance
2. Review the list of transactions in their account in the last 24 hours

## Setup the tools

In [5]:
def get_balance(account_number: str):
    if account_number == "001002":
        return "INR 51250"
    else:
        return "INR 4300"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

## Definition of tools

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get the transactions for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
]

## chat() to call Sarvam API

In [7]:
from typing import NamedTuple

class ToolCallFunction(NamedTuple):
    name: str
    arguments: str
    
class ToolCall(NamedTuple):
    id: str
    function: ToolCallFunction

In [8]:
def chat(messages: list, tools: list):
    assistant_message = ""
    
    while True:
        stream = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools,
            stream=True, # Request a stream of response chunks as and when generated by AI
        )

        
        tool_call_chunks = []
        tool_calls = []
        current_tool_call_idx = None
        
        # Iterate through each chunk in the stream
        #     chunks in a stream might represent either 
        #          a) partial AI Messages OR 
        #          b) partial Tool Call messages
        #     but never both simultanously in a single stream of chunks
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta:
                delta = chunk.choices[0].delta
                # If chunk neither contains an AI Message
                if not delta.content and not delta.tool_calls:
                    continue

                # Establish if this is a stream of AI Message chunks
                if delta.content:
                    assistant_message += delta.content
                    # stream AI Message back to caller (eventually reaches gradio)
                    yield assistant_message

                # Establish if this is a stream of Tool Call messages
                # Accumulate tool call data into tool_call_chunks
                if delta.tool_calls:
                    assert len(assistant_message) == 0, f'{assistant_message}'
                    tool_call = delta.tool_calls[0]
                    tool_call_idx = tool_call.index                    
                    # Group chunks for each distinct tool call into its own nested sub-list
                    #     e.g. 20 chunks could represent 4 tool calls (5 chunks per tool)
                    if tool_call_idx != current_tool_call_idx:
                        tool_call_chunks.append([])
                        current_tool_call_idx = tool_call_idx
                    tool_call_chunks[-1].append(tool_call)

                # No more chunks left
                if chunk.choices[0].finish_reason:
                    break

        # Merge the tool call chunks (tool_call_chunks) into tool calls (tool_calls)     
        if tool_call_chunks:
            for chunks in tool_call_chunks:
                tool_id, tool_name, tool_args = '', '', ''
                for chunk in chunks:
                    if chunk.id:
                        tool_id += chunk.id
                    if chunk.function and chunk.function.name:
                        tool_name += chunk.function.name
                    if chunk.function and chunk.function.arguments:
                        tool_args += chunk.function.arguments
                tool_call_function = ToolCallFunction(name=tool_name, arguments=tool_args)
                tool_call = ToolCall(id=tool_id, function=tool_call_function)
                tool_calls.append(tool_call)
        
        # Invoke the tools, Tool results get collected into messages
        # Loop continues sending Tool Results back to AI until it responds back with AI message for the user
        if tool_calls:
            handle_tool_calls(tool_calls, messages)

        # Break if we completed streaming back an AI message for the user
        if len(assistant_message) > 0:
            break
                
    # 3. Call AI again with updated history to process further 


In [9]:
tools_map = {
    'get_balance': get_balance,
    'get_transactions': get_transactions,
}

def invoke_tool(tool_name: str, tool_args: dict):
    result = tools_map[tool_name](**tool_args)    
    return result

In [10]:
import json
def handle_tool_calls(tool_calls: list, messages: list):
    # 2. For each tool
    for tool_call in tool_calls:
    
        # a. Get the function name and arguments
        f_name = tool_call.function.name
        f_args = json.loads(tool_call.function.arguments)
        
        # b. Invoke the function and get the result
        result = invoke_tool(f_name, f_args)
        print("Result from Tool Call :: ", result)   
        
        # c. Add the result to chat_history
        messages.append(
            {
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
                
            }
        )

## Chat with the bot

In [11]:
def run_chatbot(user_message: str, chat_history: list):

    # Gradio sends us the chat_history in a slightly different format
    # convert it into a format that Sarvam AI expects
    #      'role': 'assistant', 'metadata': None, 'content': [{'text': 'how are you', 'type': 'text'}]
    chat_history = [
        {
            "role": message["role"], 
            "content": message["content"][0]["text"]
        } for message in chat_history
    ]
    
    # Initialize chat history with System prompt
    chat_history=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ] + chat_history

          
    # Get user message and add it to chat history
    chat_history.append(
        {"role": "user", "content": user_message.strip()}
    )

    # Call chat() to get streaming AI response chunks from AI 
    # yield each chunk back to gradio
    yield from chat(chat_history, tools)

## Run the chatbot - Integrate with gradio

In [12]:
import gradio as gr

chat_interface = gr.ChatInterface(
    fn=run_chatbot, 
    title="ABC Bank - Customer Service Chat", 
)

chat_interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Result from Tool Call ::  INR 51250
Result from Tool Call ::  [('Time', 'Transaction', 'INR'), ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'), ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]
Result from Tool Call ::  INR 4300
Result from Tool Call ::  No transactions in the last 24 hrs in your account


In [ ]:
chat_interface.close()